# Chapter 4: Building the Rate Manual
### *Cyber Ratemaking and Security Controls*

Arclight has been writing one type of risk: mid-size healthcare companies that look like Meridian Medical Group. But the sales team is now fielding calls from companies that range from a 20-person law firm with no IT staff to a regional hospital network with a dedicated security operations center.

Charging everyone the same premium is not a business — it is adverse selection waiting to happen. The poorly-secured companies will flock to you because you are cheap. The well-secured ones will go elsewhere because you are expensive relative to their actual risk. You end up with a book full of bad risks at a price designed for average risks.

**Your job: build a rate manual.** Define rating factors for the security controls that most predict cyber loss frequency and severity, then price a mixed book of applicants at their individually-appropriate rates.

> **Adverse selection** is the process by which a fixed price attracts risks above the mean and repels risks below it. The solution is risk segmentation: charge each risk what it actually costs.

## The math

### Multiplicative rating model

The standard actuarial approach is a multiplicative model. Start from a **base premium** $P_0$ (the rate for a risk with all controls at their reference level), then apply a factor for each rating variable:

$$P = P_0 \times F_{\text{MFA}} \times F_{\text{EDR}} \times F_{\text{backup}} \times F_{\text{patch}} \times \cdots$$

Each factor $F_i \geq 0$ adjusts the premium up or down relative to the reference class. A factor of 1.0 means no change; 0.6 means a 40% credit; 2.5 means a 150% surcharge.

### Where do factors come from?

In a mature line, factors come from **loss experience**: fit a GLM to historical claims data regressing loss cost on control variables. In cyber, the data is still sparse, so most insurers combine a small internal dataset with vendor threat intelligence and security research.

For this chapter, factors are derived from published cyber loss studies (Verizon DBIR, CrowdStrike threat reports). They represent the **relativity** of expected loss for each control state relative to the reference class.

### Book adequacy

A rate manual is only as good as its **overall adequacy**: the weighted-average premium across the book must still cover expected losses plus expenses. If your factors are well-calibrated but your base rate is wrong, every policy is either overpriced or underpriced by the same proportional amount.

In [ ]:
# Concept — multiplicative rating factors and the premium spread they create
import numpy as np
import matplotlib.pyplot as plt

_CONTROLS = ['MFA', 'EDR', 'Backup', 'Patch']
_WORST    = np.array([1.00, 1.00, 1.00, 1.00])   # no controls (Apex / Pinecrest)
_BEST     = np.array([0.35, 0.45, 0.30, 0.50])   # full controls (BlueSky / Luminary)

# All-applicant data: (name, revenue_M, freq_multiplier)
_APPLICANTS = [
    ('Meridian',   10, 0.65 * 0.70 * 0.60 * 0.75),
    ('Apex Law',    5, 1.00 * 1.00 * 1.00 * 1.00),
    ('BlueSky',    25, 0.35 * 0.45 * 0.30 * 0.50),
    ('Crestwood',   8, 1.00 * 0.70 * 0.60 * 0.75),
    ('Harborview', 15, 0.65 * 0.45 * 0.60 * 0.50),
    ('Irongate',   30, 1.00 * 1.00 * 0.60 * 1.00),
    ('Luminary',   12, 0.35 * 0.45 * 0.30 * 0.50),
    ('Oakdale',    20, 0.65 * 0.70 * 0.30 * 0.75),
    ('Pinecrest',   3, 1.00 * 1.00 * 1.00 * 1.00),
    ('Quorum',     18, 0.35 * 0.70 * 0.60 * 0.50),
]
_BASE_PURE_PER_M = (0.08*0.025 + 0.04*0.020 + 0.10*0.004) * 1_000_000   # $3,200/M revenue
_EXPENSE         = 0.35
_gross = [(n, rev * _BASE_PURE_PER_M * fm / (1 - _EXPENSE))
          for n, rev, fm in _APPLICANTS]
_gross_sorted = sorted(_gross, key=lambda x: x[1])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Multiplicative Rating Factors — Why Segmentation Matters',
             fontsize=12, fontweight='bold')

# Left: individual factor values, worst vs best
_x = np.arange(len(_CONTROLS))
_w = 0.35
ax1.bar(_x - _w/2, _WORST, _w, color='#e74c3c', alpha=0.85, label='No controls (Apex / Pinecrest)')
ax1.bar(_x + _w/2, _BEST,  _w, color='#27ae60', alpha=0.85, label='Full controls (BlueSky / Luminary)')
ax1.axhline(1.0, color='black', linewidth=0.8, linestyle='--', alpha=0.4)

_prod_worst = float(np.prod(_WORST))
_prod_best  = float(np.prod(_BEST))
ax1.annotate(
    f'Combined multiplier: {_prod_worst:.2f}x\n(reference class)',
    xy=(3.35, _WORST[3]), xytext=(2.5, 1.18),
    fontsize=8, color='#c0392b', ha='center',
    arrowprops=dict(arrowstyle='->', color='#c0392b', lw=1.0),
    bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#c0392b', alpha=0.92))
ax1.annotate(
    f'Combined multiplier: {_prod_best:.3f}x\n({_prod_best:.1%} of reference class)',
    xy=(3.35, _BEST[3]), xytext=(1.5, 0.62),
    fontsize=8, color='#1e8449', ha='center',
    arrowprops=dict(arrowstyle='->', color='#1e8449', lw=1.0),
    bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#1e8449', alpha=0.92))

ax1.set_xticks(_x)
ax1.set_xticklabels(_CONTROLS, fontsize=11)
ax1.set_ylabel('Frequency factor  (1.0 = uncontrolled reference class)', fontsize=9)
ax1.set_title('Individual factors multiply together', fontsize=10)
ax1.legend(fontsize=8.5, framealpha=0.9)
ax1.set_ylim(0, 1.38)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Right: gross premium by applicant (sorted)
_names    = [r[0] for r in _gross_sorted]
_premiums = [r[1] for r in _gross_sorted]
_colors   = ['#27ae60' if p < 20_000 else ('#e67e22' if p < 60_000 else '#e74c3c')
             for p in _premiums]
_bars = ax2.barh(_names, _premiums, color=_colors, alpha=0.85, edgecolor='white')
ax2.set_xlabel('Indicated gross premium (per policy, $10M rev equivalent)', fontsize=9)
ax2.set_title('Premium spread across applicant book\nSame flat rate would cause adverse selection', fontsize=10)
ax2.xaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _: f'${v/1_000:.0f}K'))
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.grid(True, axis='x', alpha=0.3)

# Annotate the ratio
_ratio = _premiums[-1] / _premiums[0]
ax2.text(_premiums[-1] * 1.02, len(_names) - 1,
         f'{_ratio:.0f}x spread', fontsize=9, color='#e74c3c',
         va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Best-in-class combined multiplier:  {_prod_best:.4f}x  ({_prod_best:.1%} of reference)')
print(f'Worst-in-class combined multiplier: {_prod_worst:.2f}x')
print(f'Ratio: {_prod_worst/_prod_best:.0f}x difference in expected frequency')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

# Base exposure parameters (same as previous chapters)
BASE_ANNUAL_REVENUE = 10_000_000
EXPENSE_RATIO       = 0.35

# Base threat frequencies and severities (reference class: all controls at worst level)
# These are the rates before any control credits are applied
BASE_THREATS = [
    ('Ransomware',                0.18, 0.0280),   # (name, base_freq, severity_pct)
    ('Data Breach',               0.10, 0.0220),
    ('Business Email Compromise', 0.20, 0.0042),
]

FREQ_BASE = np.array([t[1] for t in BASE_THREATS])
SEV_BASE  = np.array([t[2] * BASE_ANNUAL_REVENUE for t in BASE_THREATS])
PURE_BASE = float((FREQ_BASE * SEV_BASE).sum())

print('Base pure premium (worst-class risk): ${:,.0f}'.format(PURE_BASE))
print('Base gross premium:                   ${:,.0f}'.format(PURE_BASE / (1 - EXPENSE_RATIO)))

## The rating factors

Four controls with the strongest predictive relationship to cyber loss costs, each with three tiers. The **reference class** (factor = 1.0) is the worst tier for each control — factors below 1.0 are credits for better security posture.

Factors apply to **frequency only** — this is a simplification. In practice, controls like encryption and backup maturity also affect severity (an encrypted backup means you may not need to pay the ransom). That complexity is left for Chapter 4's extension.

In [ ]:
# Rating factors by control and tier
# Each entry: (control_name, tier_name, freq_factor, description)
RATING_FACTORS = {
    'MFA': [
        ('None',         1.00, 'No multi-factor authentication anywhere'),
        ('Partial',      0.65, 'MFA on email and VPN only'),
        ('Full',         0.35, 'MFA enforced across all systems and privileged access'),
    ],
    'EDR': [
        ('None',         1.00, 'No endpoint detection and response tooling'),
        ('Basic',        0.70, 'Antivirus/EPP only, no behavioral detection'),
        ('Advanced',     0.45, 'EDR with 24/7 managed detection and response (MDR)'),
    ],
    'Backup': [
        ('None',         1.00, 'No tested backups or backups connected to production network'),
        ('Basic',        0.60, 'Regular backups, not regularly tested, not fully air-gapped'),
        ('Immutable',    0.30, 'Immutable off-site backups, tested quarterly'),
    ],
    'Patch': [
        ('>90 days',     1.00, 'Critical patches applied > 90 days after release'),
        ('30-90 days',   0.75, 'Critical patches applied within 30-90 days'),
        ('<30 days',     0.50, 'Critical patches applied within 30 days'),
    ],
}

# Display the factor table
rows = []
for control, tiers in RATING_FACTORS.items():
    for tier_name, factor, desc in tiers:
        rows.append({'Control': control, 'Tier': tier_name,
                     'Freq. Factor': factor, 'Description': desc})
tbl = pd.DataFrame(rows)
display(tbl.to_html(index=False))

## The applicant book

Ten companies have applied for coverage. Each has a different security control profile. The underwriting questionnaire has been scored — now price each one.

In [ ]:
# Applicant profiles: (name, revenue_M, MFA_tier, EDR_tier, Backup_tier, Patch_tier)
APPLICANTS = [
    ('Meridian Medical',      10, 'Partial',  'Basic',    'Basic',     '30-90 days'),
    ('Apex Law Group',         5, 'None',     'None',     'None',      '>90 days'),
    ('BlueSky Logistics',     25, 'Full',     'Advanced', 'Immutable', '<30 days'),
    ('Crestwood Schools',      8, 'None',     'Basic',    'Basic',     '30-90 days'),
    ('Harborview Clinic',     15, 'Partial',  'Advanced', 'Basic',     '<30 days'),
    ('Irongate Manufacturing', 30, 'None',    'None',     'Basic',     '>90 days'),
    ('Luminary Tech',         12, 'Full',     'Advanced', 'Immutable', '<30 days'),
    ('Oakdale Credit Union',  20, 'Partial',  'Basic',    'Immutable', '30-90 days'),
    ('Pinecrest Dental',       3, 'None',     'None',     'None',      '>90 days'),
    ('Quorum Analytics',      18, 'Full',     'Basic',    'Basic',     '<30 days'),
]


def get_factor(control, tier):
    for t_name, f, _ in RATING_FACTORS[control]:
        if t_name == tier:
            return f
    raise ValueError(f'Unknown tier {tier} for {control}')


def price_applicant(revenue_m, mfa, edr, backup, patch):
    revenue = revenue_m * 1_000_000
    freq_multiplier = (get_factor('MFA', mfa) * get_factor('EDR', edr) *
                       get_factor('Backup', backup) * get_factor('Patch', patch))
    adj_freq = FREQ_BASE * freq_multiplier
    adj_sev  = SEV_BASE * (revenue / BASE_ANNUAL_REVENUE)   # scale severity to actual revenue
    pure     = float((adj_freq * adj_sev).sum())
    gross    = pure / (1 - EXPENSE_RATIO)
    return pure, gross, freq_multiplier


rows = []
for name, rev_m, mfa, edr, backup, patch in APPLICANTS:
    pure, gross, fmult = price_applicant(rev_m, mfa, edr, backup, patch)
    rows.append({
        'Applicant':       name,
        'Revenue':         f'${rev_m}M',
        'MFA':             mfa,
        'EDR':             edr,
        'Backup':          backup,
        'Patch':           patch,
        'Freq. Multiplier':f'{fmult:.3f}',
        'Pure Premium':    f'${pure:,.0f}',
        'Gross Premium':   f'${gross:,.0f}',
    })

applicant_df = pd.DataFrame(rows)
display(applicant_df.to_html(index=False))

## Your turn: set the base rate and write the book

The base rate above is derived from the factor table. Now decide:

1. Which applicants do you **accept** at the modeled rate (check the box)?
2. Do you want to adjust the **base rate multiplier** to change overall book adequacy?
3. Simulate a year and see your actual loss ratio versus the modeled expectation.

Watch for adverse selection: if you accept only the high-risk applicants (or if you underprice), your actual loss ratio will exceed your modeled one.

In [ ]:
# Pre-compute modeled values for all applicants
applicant_data = []
for name, rev_m, mfa, edr, backup, patch in APPLICANTS:
    pure, gross, fmult = price_applicant(rev_m, mfa, edr, backup, patch)
    freq_adj = FREQ_BASE * fmult
    sev_adj  = SEV_BASE * (rev_m * 1_000_000 / BASE_ANNUAL_REVENUE)
    applicant_data.append({
        'name':      name,
        'revenue':   rev_m * 1_000_000,
        'pure':      pure,
        'gross':     gross,
        'fmult':     fmult,
        'freq_adj':  freq_adj,
        'sev_adj':   sev_adj,
    })

# Widgets: one checkbox per applicant
checkboxes = [
    widgets.Checkbox(value=True, description=f"{a['name']} (${a['gross']:,.0f})",
                     layout=widgets.Layout(width='360px'))
    for a in applicant_data
]
base_mult_slider = widgets.FloatSlider(
    value=1.0, min=0.5, max=2.0, step=0.05,
    description='Base rate multiplier:',
    readout_format='.2f',
    layout=widgets.Layout(width='500px')
)
n_sims_slider = widgets.IntSlider(
    value=500, min=100, max=2000, step=100,
    description='Sim years:',
    layout=widgets.Layout(width='400px')
)
run_button = widgets.Button(description='Write the book', button_style='primary')
output     = widgets.Output()

cb_grid = widgets.GridBox(
    checkboxes,
    layout=widgets.Layout(grid_template_columns='repeat(2, 380px)')
)


def run_sim(_):
    base_mult  = base_mult_slider.value
    n_sims     = n_sims_slider.value
    rng        = np.random.default_rng()

    accepted = [a for a, cb in zip(applicant_data, checkboxes) if cb.value]
    if not accepted:
        with output:
            output.clear_output(wait=True)
            display(HTML('<p style="color:crimson">No applicants accepted. Select at least one.</p>'))
        return

    total_modeled_pure  = sum(a['pure']  for a in accepted)
    total_modeled_gross = sum(a['gross'] for a in accepted) * base_mult
    total_premium       = total_modeled_gross   # premium charged = gross * base_mult

    # Simulate n_sims years
    annual_claims = np.zeros(n_sims)
    for a in accepted:
        fires = rng.random((n_sims, len(a['freq_adj']))) < a['freq_adj']
        annual_claims += (fires * a['sev_adj']).sum(axis=1)

    annual_premiums = total_premium
    loss_ratios     = annual_claims / annual_premiums
    combined_ratios = (annual_claims + annual_premiums * EXPENSE_RATIO) / annual_premiums
    uw_incomes      = annual_premiums - annual_claims - annual_premiums * EXPENSE_RATIO

    mean_lr   = loss_ratios.mean()
    std_lr    = loss_ratios.std()
    p95_lr    = np.percentile(loss_ratios, 95)
    loss_prob = (loss_ratios > 1.0).mean()
    modeled_lr = total_modeled_pure / total_premium

    with output:
        output.clear_output(wait=True)

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        # Per-applicant premium bar chart
        names    = [a['name'].split()[0] for a in accepted]   # first word only for brevity
        premiums = [a['gross'] * base_mult for a in accepted]
        pures    = [a['pure']  for a in accepted]
        x        = np.arange(len(accepted))
        axes[0].bar(x, premiums, color='steelblue', label='Charged premium')
        axes[0].bar(x, pures,    color='crimson',   alpha=0.6, label='Pure premium (expected cost)')
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(names, rotation=45, ha='right', fontsize=8)
        axes[0].set_title('Charged vs. expected cost per policy')
        axes[0].set_ylabel('$')
        axes[0].legend(fontsize=8)
        axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}'))

        # Loss ratio histogram
        axes[1].hist(loss_ratios, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
        axes[1].axvline(mean_lr,    color='black',      linestyle='--', linewidth=1.5,
                        label=f'Simulated mean {mean_lr:.1%}')
        axes[1].axvline(modeled_lr, color='seagreen',   linestyle='--', linewidth=1.5,
                        label=f'Modeled LR {modeled_lr:.1%}')
        axes[1].axvline(1.0,        color='darkorange', linestyle=':',  linewidth=1.5,
                        label='Loss ratio = 100%')
        axes[1].set_title(f'Portfolio loss ratio — {len(accepted)} policies, {n_sims} years')
        axes[1].set_xlabel('Loss ratio')
        axes[1].set_ylabel('Frequency')
        axes[1].legend(fontsize=8)

        # Freq multiplier scatter: adequacy per policy
        fmults  = [a['fmult']           for a in accepted]
        margins = [(a['gross'] * base_mult - a['pure']) / (a['gross'] * base_mult)
                   for a in accepted]
        scatter_colors = ['seagreen' if m > 0 else 'crimson' for m in margins]
        axes[2].scatter(fmults, margins, c=scatter_colors, s=80, zorder=3)
        for i, a in enumerate(accepted):
            axes[2].annotate(a['name'].split()[0], (fmults[i], margins[i]),
                             fontsize=7, xytext=(4, 4), textcoords='offset points')
        axes[2].axhline(0, color='black', linewidth=0.8)
        axes[2].set_xlabel('Frequency multiplier (higher = riskier applicant)')
        axes[2].set_ylabel('Profit margin')
        axes[2].set_title('Profit margin vs. risk level')
        axes[2].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        # Adequacy summary
        lines = []
        lines.append(f'<p><b>{len(accepted)} policies accepted | '
                     f'Total annual premium: ${total_premium:,.0f} | '
                     f'Base rate multiplier: {base_mult:.2f}x</b></p>')
        lines.append(f'<p>Modeled loss ratio: <b>{modeled_lr:.1%}</b> &nbsp;|&nbsp; '
                     f'Simulated mean: <b>{mean_lr:.1%}</b> &nbsp;|&nbsp; '
                     f'Std dev: <b>{std_lr:.1%}</b> &nbsp;|&nbsp; '
                     f'95th pct: <b>{p95_lr:.1%}</b></p>')
        lines.append(f'<p>Loss years (loss ratio &gt; 100%): <b>{loss_prob:.1%}</b></p>')

        deviation = mean_lr - modeled_lr
        if abs(deviation) < 0.03:
            lines.append(f'<p style="color:seagreen"><b>Well-calibrated.</b> '
                         f'The simulated mean ({mean_lr:.1%}) tracks the modeled loss ratio '
                         f'({modeled_lr:.1%}) closely. Your factors are doing their job.</p>')
        elif deviation > 0.05:
            lines.append(f'<p style="color:crimson"><b>Adverse selection detected.</b> '
                         f'Simulated loss ratio ({mean_lr:.1%}) exceeds modeled ({modeled_lr:.1%}) '
                         f'by {deviation:.1%}. You have likely accepted a disproportionate share '
                         f'of high-risk applicants or underpriced via the base rate multiplier.</p>')
        else:
            lines.append(f'<p style="color:darkorange"><b>Slight drift.</b> '
                         f'Simulated mean is {deviation:+.1%} versus modeled. '
                         f'Check which applicants you accepted and review the base rate multiplier.</p>')

        if base_mult < 0.85:
            lines.append(f'<p style="color:crimson"><b>Underpriced book.</b> '
                         f'A base rate multiplier of {base_mult:.2f}x means you are charging '
                         f'{(1-base_mult):.0%} less than the modeled gross rate. '
                         f'This is fine only if you believe your factors overestimate risk.</p>')
        elif base_mult > 1.30:
            lines.append(f'<p style="color:darkorange"><b>Overpriced book.</b> '
                         f'At {base_mult:.2f}x, you may lose well-secured risks to competitors '
                         f'who price more accurately. Adverse selection runs both ways.</p>')

        lines.append('<hr><p><b>Key takeaway.</b> '
                     'A flat rate attracts bad risks and repels good ones. '
                     'Rating factors segment risk so each policy pays for its own expected cost. '
                     'The base rate multiplier controls overall adequacy: too low and the book '
                     'loses money; too high and good risks leave. '
                     'In cyber, the spread between best-in-class (full MFA + MDR + immutable backup) '
                     'and worst-in-class can easily be a 15-20x difference in expected loss.</p>')
        display(HTML(''.join(lines).replace('$', '&#36;')))


run_button.on_click(run_sim)
display(widgets.VBox([
    widgets.HTML('<b>Select applicants to accept:</b>'),
    cb_grid,
    base_mult_slider,
    n_sims_slider,
    run_button,
    output
]))

## Hints

<details><summary>Hint 1 — adverse selection experiment</summary>

Accept only the three worst-rated applicants (Apex Law, Irongate, Pinecrest) at the flat Meridian rate (\$55K from Chapter 1). Run the simulation. Compare the modeled loss ratio vs. the simulated mean.

</details>

<details><summary>Hint 2 — the spread between best and worst</summary>

Look at the frequency multipliers in the applicant table. BlueSky Logistics and Luminary Tech have full controls at all tiers. Apex Law Group and Pinecrest Dental have no controls. Compute the ratio of their gross premiums — that is the range your rate manual spans.

</details>

<details><summary>Hint 3 — base rate adequacy</summary>

The "profit margin vs. risk level" chart shows each accepted policy's margin. Any policy with a negative margin (red dot) is underpriced individually, even if the overall book is adequate. Set the base multiplier to 0.7 and re-run to see a structurally inadequate book.

</details>

<details><summary>Hint 4 — real-world factor development</summary>

These factors are illustrative. In practice, factors are developed by fitting a generalized linear model (GLM) to historical claim data, typically a Tweedie or Poisson GLM for frequency and a gamma GLM for severity. The factors are the exponentiated coefficients. As of 2025, most cyber insurers are still in the early stages of building datasets large enough to credibly estimate factors purely from their own experience.

</details>

## What's next

**Chapter 5 — The Long Tail (Reserving and IBNR).** You have been pricing and collecting premiums, but claims do not arrive fully formed. A data breach that occurs in October may not be reported until January, and the full cost — forensics, litigation, regulatory fines — may not be known for two or three years.

This is the **tail problem**. At any given moment, your balance sheet carries incurred-but-not-reported (IBNR) liabilities that are real but invisible. Setting reserves too low makes the company look more profitable than it is; too high and you tie up capital unnecessarily. Chapter 5 introduces the chain-ladder method: the actuarial workhorse for estimating ultimate losses from a triangle of partial data.